# 🍃 大白菜叶色表型提取 — 交互式演示

本 Notebook 逐步演示:
1. 图像预处理与颜色校准
2. 叶片分割 (多种方法对比)
3. 多颜色空间特征提取
4. GWAS表型表生成
5. 性状可视化与质控

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

## 1. 加载并检查叶色表型数据 (假设已运行过批量提取)

In [ ]:
# 读取表型表
df = pd.read_csv('../output/leaf_color_phenotypes.csv', index_col=0)
print(f'样本数: {df.shape[0]}')
print(f'性状数: {df.shape[1]}')
df.head()

## 2. 关键性状分布检查

In [ ]:
key_traits = ['CIELAB_L_mean', 'CIELAB_A_mean', 'CIELAB_B_mean',
              'GLI', 'DGCI', 'CIELAB_greenness']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, trait in zip(axes.flat, key_traits):
    if trait in df.columns:
        ax.hist(df[trait].dropna(), bins=30, edgecolor='black', alpha=0.7)
        ax.set_title(trait)
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 3. 遗传力评估 — 筛选高遗传力性状

计算每个性状在重复间的广义遗传力 (broad-sense heritability):

$$H^2 = \frac{\sigma^2_g}{\sigma^2_g + \sigma^2_e/n}$$

只保留 $H^2 > 0.5$ 的性状进行GWAS分析。

In [ ]:
def calculate_heritability(df_long, trait_col, id_col='sample_id'):
    """
    计算广义遗传力 H² = Vg / (Vg + Ve/n)
    需要原始(未汇总)数据, 包含每重复的测量值。
    """
    import statsmodels.api as sm
    from statsmodels.formula.api import mixedlm
    
    # 拟合混合线性模型: trait ~ (1|sample_id)
    try:
        model = mixedlm(f"{trait_col} ~ 1", df_long, groups=df_long[id_col])
        result = model.fit(reml=True)
        vg = result.cov_re.iloc[0, 0]  # 遗传方差
        ve = result.scale                 # 残差方差
        n = df_long.groupby(id_col).size().mean()  # 平均重复数
        H2 = vg / (vg + ve / n)
        return max(0, min(1, H2))
    except Exception:
        return np.nan

# 如果有原始(未汇总)数据, 可运行如下:
# df_raw = pd.read_csv('../output/leaf_color_phenotypes_raw.csv')
# trait_cols = [c for c in df_raw.columns if c not in ('sample_id', 'image_path', 'replicate')]
# heritabilities = {trait: calculate_heritability(df_raw, trait) for trait in trait_cols}
# h2_df = pd.Series(heritabilities).sort_values(ascending=False)
# print('Top 20 高遗传力性状:')
# print(h2_df.head(20))

## 4. 性状间相关性矩阵

识别冗余性状, 为每个性状类群选择一个代表性性状。

In [ ]:
# 筛选数值型且有足够方差的性状
num_cols = df.select_dtypes(include=[np.number]).columns
var_cols = [c for c in num_cols if df[c].std() > 1e-6]
corr = df[var_cols].corr()

# 只为关键性状绘制热图
key_all = [c for c in df.columns if any(
    c.startswith(p) for p in ['CIELAB_L', 'CIELAB_A', 'CIELAB_B', 
                               'GLI', 'VARI', 'DGCI', 'HSV_H', 'HSV_S']
) and c.endswith('_mean')]
if len(key_all) > 1:
    plt.figure(figsize=(12, 10))
    sns.heatmap(df[key_all].corr(), annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1)
    plt.title('关键性状相关性矩阵')
    plt.tight_layout()
    plt.show()

## 5. PCA分析 — 叶色表型空间的群体结构

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 选择代表性性状子集
repr_traits = [c for c in df.columns if '_mean' in c and not '_std' in c and not '_cv' in c]
repr_traits = [c for c in repr_traits if c in df.columns and df[c].std() > 0]

X = StandardScaler().fit_transform(df[repr_traits].dropna())
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7, s=50)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('叶色表型 PCA — 群体结构')
plt.show()

print('Top 5 PC1 loading traits:')
loadings = pd.Series(pca.components_[0], index=repr_traits).abs().sort_values(ascending=False)
print(loadings.head(10))

## 6. 叶色聚类 — 无监督验证分级标准

In [ ]:
from sklearn.cluster import KMeans

# 选择CIELAB a* 和 b* 作为叶色分类特征
lab_data = df[['CIELAB_A_mean', 'CIELAB_B_mean']].dropna()

# K-means聚类 (尝试k=3~5)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [3, 4, 5]):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(lab_data)
    for i in range(k):
        ax.scatter(lab_data.iloc[labels == i, 0],
                   lab_data.iloc[labels == i, 1],
                   label=f'Cluster {i+1}', alpha=0.7, s=40)
    ax.set_xlabel('a* (Red−Green)')
    ax.set_ylabel('b* (Yellow−Blue)')
    ax.set_title(f'K-Means (k={k})')
    ax.legend()
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.suptitle('大白菜叶色 CIELAB a*-b* 聚类', y=1.02, fontsize=14)
plt.show()

## 总结

提取的叶色表型数据可直接用于:
1. **GWAS分析**: 将 `_mean` 列作为表型值, 输入 GAPIT3 / rMVP
2. **遗传力过滤**: 只保留 H² > 0.5 的性状
3. **表型聚类**: 与人工叶色分级标准交叉验证
4. **多性状GWAS**: 对相关性状组进行多变量GWAS (mvGWAS)